In [2]:
import numpy as np
from qdk_chemistry.data import Element, Structure, MajoranaMapping
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

coords_ang = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.740848],
    ]
)
structure = Structure(coords_ang, [Element.H, Element.H])

scf = create("scf_solver")
scf_energy, wavefunction = scf.run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)

# 2) Electronic Hamiltonian
ham_constructor = create("hamiltonian_constructor")
hamiltonian = ham_constructor.run(wavefunction.get_orbitals())

# 3) Qubit Hamiltonian
mapper = create("qubit_mapper")
n_spin_orbitals = hamiltonian.get_orbitals().get_num_molecular_orbitals() * 2
qubit_hamiltonian = mapper.run(hamiltonian, MajoranaMapping.jordan_wigner(n_spin_orbitals))

# 4) Ground-state energy of the qubit Hamiltonian
solver = create("qubit_hamiltonian_solver")
qubit_energy, ground_state = solver.run(qubit_hamiltonian)
qubit_energy = qubit_energy

In [3]:
import scipy
import math
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter import Trotter
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter_error import trotter_steps_commutator
import numpy as np

_PAULI = {
    "I": np.eye(2, dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def _pauli_matrix(pauli_map: dict[int, str], num_qubits: int) -> np.ndarray:
    """Build the full 2^n x 2^n matrix for a sparse Pauli map {qubit_idx: 'X'/'Y'/'Z'}."""
    mat = np.array([[1.0]], dtype=complex)
    for q in range(num_qubits - 1, -1, -1):
        mat = np.kron(mat, _PAULI[pauli_map.get(q, "I")])
    return mat

def numerical_error_sweep(hamiltonian, t=0.1, num_divisions=1, order=1):
    num_qubits = hamiltonian.num_qubits
    builder = Trotter(num_divisions=num_divisions, time=t, order=order)
    unitary = builder.run(hamiltonian)
    container = unitary.get_container()

    # Build Trotter unitary matrix
    dim = 2**num_qubits
    u_step = np.eye(dim, dtype=complex)
    for term in container.step_terms:
        pauli_mat = _pauli_matrix(term.pauli_term, num_qubits)
        u_step = scipy.linalg.expm(-1j * term.angle * pauli_mat) @ u_step
    u_trot = np.linalg.matrix_power(u_step, container.step_reps)

    # Exact unitary
    hamiltonian_matrix = hamiltonian.to_matrix()
    u_exact = scipy.linalg.expm(-1j * t * hamiltonian_matrix)

    # State-specific error: ||U_trot|psi0> - U_exact|psi0>||_2
    psi_trot = u_trot @ ground_state
    psi_exact = u_exact @ ground_state
    ground_state_error = np.linalg.norm(psi_trot - psi_exact)

    # Operator error: ||U_trot - U_exact||_2
    operator_error = np.linalg.norm(u_trot - u_exact)

    # Diagonalize u_trot to extract ground state energy
    eigvals = np.linalg.eigvals(u_trot)
    energies = -np.angle(eigvals) / t
    u_trot_ground = np.min(energies.real)

    return ground_state_error, operator_error, u_trot_ground

t = math.pi/qubit_hamiltonian.schatten_norm
target_accuracy = 1e-2

rows = []
for order in [1, 2]:
    for num_phase_bit in range(6, 11, 2):
        scaled_t = t * 2**(num_phase_bit - 1)
        step = trotter_steps_commutator(qubit_hamiltonian, scaled_t, target_accuracy, order=order)
        for num_divisions in range(1, 100, 10):
            ground_state_error, operator_error, u_trot_ground = numerical_error_sweep(qubit_hamiltonian, t=scaled_t, num_divisions=num_divisions, order=order)
            rows.append({
                "order": order,
                "num_phase_bit": num_phase_bit,
                "t": t,
                "scaled_t": scaled_t,
                "num_divisions": num_divisions,
                "operator_error": operator_error,
                "ground_state_error": ground_state_error,
                "u_trot_ground": u_trot_ground,
            })

import pandas as pd
df = pd.DataFrame(rows)
df


t=0.9959363364087607, qubit_hamiltonian.schatten_norm=3.154411119206714


,order,num_phase_bit,t,scaled_t,num_divisions,operator_error,ground_state_error,u_trot_ground
0,1,6,0.995936,31.869963,1,5.305509,1.458720,-0.090754
1,1,6,0.995936,31.869963,11,1.128357,0.675477,-0.098251
2,1,6,0.995936,31.869963,21,1.026404,0.690729,-0.090754
3,1,6,0.995936,31.869963,31,0.422909,0.258123,-0.090754
4,1,6,0.995936,31.869963,41,0.257304,0.141654,-0.090754
5,1,6,0.995936,31.869963,51,0.183597,0.091800,-0.090754
6,1,6,0.995936,31.869963,61,0.142734,0.065543,-0.090754
7,1,6,0.995936,31.869963,71,0.116950,0.049903,-0.090754
8,1,6,0.995936,31.869963,81,0.099230,0.039778,-0.090754
9,1,6,0.995936,31.869963,91,0.086299,0.032812,-0.090754


## Old code for resource estimate the circuit
This cell takes about 40 secs

In [ ]:
from utils.qpe_utils import compute_evolution_time
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
import pandas as pd


def run_trotter(m_precision, trotter_order, target_accuracy=1e-2):
    evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=m_precision)
    unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "trotter", time=evolution_time, order=trotter_order, power_strategy="rescale", target_accuracy=target_accuracy)
    circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")
    circuit_builder = create(
        "qpe_circuit_builder", 
        "qdk_iterative",
        num_bits=m_precision,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=circuit_mapper,
    )
    state_prep = identity_state_prep(num_qubits=qubit_hamiltonian.num_qubits)
    iqpe_iter_circuits = circuit_builder.run(
        state_preparation=state_prep,
        qubit_hamiltonian=qubit_hamiltonian,
    )
    largest_circuit = iqpe_iter_circuits[0]
    result = largest_circuit.estimate()
    logical_estimate = result["logicalCounts"]

    return logical_estimate, target_accuracy, largest_circuit

rows = []
for m_precision in [6, 8, 10]:
    for trotter_order in [1, 2]:
        logical_estimate, target_accuracy, largest_circuit = run_trotter(m_precision, trotter_order)
        rows.append({
            "m_precision": m_precision,
            "trotter_order": trotter_order,
            "target_accuracy": target_accuracy,
            "numQubits": logical_estimate["numQubits"],
            "rotationCount": logical_estimate["rotationCount"],
            "rotationDepth": logical_estimate["rotationDepth"],
        })

df = pd.DataFrame(rows)
df


In [ ]:
from qdk.widgets import Circuit

Circuit(largest_circuit.get_qsharp_circuit())